# Seed / Variance Validation  —  Bootstrap CI on PLUSVein RL850

**What this does (and does not) measure.** This notebook reuses your existing
`plusvein_eval.py` (head-stripped embeddings + cosine verification, inference only,
**no retraining**) and adds a **bootstrap** over the verification pairs to get a
confidence interval on EER and AUC for every model and for the score-level
ensemble. It also runs a paired bootstrap significance test of the ensemble
against the best single model.

This is **test-set sampling variance** (bootstrap CI over evaluation pairs), a
standard, defensible quantity to report. It is **not** multi-seed training
variance — that would require retraining each model several times. Word the
paper accordingly (suggested text is printed at the end).

The per-model point EER/AUC will match `plusvein_eer_summary.csv` exactly,
because the same `compute_eer` and the same pairs are used. First run embeds all
images once and caches the scores to disk, so re-runs are instant.

In [ ]:
# Cell 1 — imports. Run from the SAME folder as plusvein_eval.py (the project root).
import os, sys, gc, random, itertools
from pathlib import Path
import numpy as np
from scipy import stats
import torch
from PIL import Image
from tqdm import tqdm

# import your eval module WITHOUT running training (main() only runs under __main__)
import plusvein_eval as pe
print("Loaded plusvein_eval | device:", pe.DEVICE)
print("Models:", list(pe.MODEL_REGISTRY.keys()))

In [ ]:
# Cell 2 — build the SAME genuine/impostor pair lists plusvein_eval uses.
img_paths = sorted(Path(pe.PLUSVEIN_PREPROCESSED).glob('*.png'))
assert img_paths, f"No images at {pe.PLUSVEIN_PREPROCESSED}"
identities = [pe.parse_identity(p.name) for p in img_paths]
uniq = sorted(set(identities))
id_to_idx = {k: i for i, k in enumerate(uniq)}
labels = np.array([id_to_idx[i] for i in identities])

idx_by_id = {}
for i, lab in enumerate(labels):
    idx_by_id.setdefault(lab, []).append(i)

genuine_pairs = []
for lab, idxs in idx_by_id.items():
    genuine_pairs += list(itertools.combinations(idxs, 2))

all_pairs = list(itertools.combinations(range(len(img_paths)), 2))
genuine_set = set(genuine_pairs)
impostor_pairs = [p for p in all_pairs if p not in genuine_set]

random.seed(pe.SEED)   # reproduce the SAME impostor sample as your original run
if len(impostor_pairs) > pe.MAX_IMPOSTOR_PAIRS:
    impostor_pairs = random.sample(impostor_pairs, pe.MAX_IMPOSTOR_PAIRS)

gi = np.array(genuine_pairs)
ii = np.array(impostor_pairs)
print(f"images {len(img_paths)} | hand-identities {len(uniq)} | "
      f"genuine {len(gi)} | impostor {len(ii)}")

In [ ]:
# Cell 3 — embed each model once, cache genuine/impostor cosine score arrays.
SCORES_DIR = os.path.join(pe.OUT_DIR, 'bootstrap_scores')
os.makedirs(SCORES_DIR, exist_ok=True)

def model_scores(name):
    gp = os.path.join(SCORES_DIR, f'{name}_gen.npy')
    ip = os.path.join(SCORES_DIR, f'{name}_imp.npy')
    if os.path.exists(gp) and os.path.exists(ip):
        return np.load(gp), np.load(ip)
    model = pe.load_model(name)
    if model is None:
        return None, None
    feat = pe.to_feature_extractor(name, model)
    tfm = pe.transform_for(name)
    embs = []
    with torch.no_grad():
        for p in tqdm(img_paths, desc=f'{name} embed', leave=False):
            x = tfm(Image.open(p).convert('RGB')).unsqueeze(0).to(pe.DEVICE)
            with torch.autocast('cuda', enabled=torch.cuda.is_available()):
                e = feat(x)
            embs.append(e.float().squeeze(0).cpu().numpy())
    embs = np.stack(embs)
    sim = pe.cosine_matrix(embs)
    gen = sim[gi[:, 0], gi[:, 1]].astype(np.float32)
    imp = sim[ii[:, 0], ii[:, 1]].astype(np.float32)
    np.save(gp, gen); np.save(ip, imp)
    del model, embs, sim; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return gen, imp

scores = {}
for name in pe.MODEL_REGISTRY:
    g, i = model_scores(name)
    if g is not None:
        scores[name] = (g, i)
        print(f"{name:18s} cached  (gen {len(g)}, imp {len(i)})")
    else:
        print(f"{name:18s} SKIPPED (missing checkpoint)")

In [ ]:
# Cell 4 — bootstrap CI per model + ensemble, paired significance, LaTeX output.
N_BOOT = 2000
rng = np.random.default_rng(0)
present = [m for m in pe.MODEL_REGISTRY if m in scores]

def eer_auc_fast(gen, imp):
    y = np.r_[np.ones(len(gen)), np.zeros(len(imp))]
    s = np.r_[gen, imp]
    o = np.argsort(-s, kind='mergesort'); y = y[o]
    P = y.sum(); N = len(y) - P
    tpr = np.cumsum(y) / P; fpr = np.cumsum(1 - y) / N; fnr = 1 - tpr
    k = int(np.argmin(np.abs(fnr - fpr)))
    eer = (fnr[k] + fpr[k]) / 2 * 100
    a = np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) * 0.5)
    return eer, a

print("== Per-model: point [bootstrap mean +/- std] ==")
latex_rows = []
for m in present:
    g, i = scores[m]
    e0, a0 = pe.compute_eer(g, i)[:2]          # matches your CSV
    es = np.empty(N_BOOT); as_ = np.empty(N_BOOT)
    for b in range(N_BOOT):
        gg = g[rng.integers(0, len(g), len(g))]
        ic = i[rng.integers(0, len(i), len(i))]
        es[b], as_[b] = eer_auc_fast(gg, ic)
    s_e, s_a = es.std(ddof=1), as_.std(ddof=1)
    print(f"{m:18s} EER {e0:5.2f} [{es.mean():.2f}+/-{s_e:.2f}]   "
          f"AUC {a0:.4f} [{as_.mean():.4f}+/-{s_a:.4f}]")
    latex_rows.append(f"{m} & ${e0:.2f} \\pm {s_e:.2f}$ & ${a0:.4f} \\pm {s_a:.4f}$ \\\\")

# ensemble = mean cosine score across models, per pair (score-level fusion)
gE = np.mean([scores[m][0] for m in present], axis=0)
iE = np.mean([scores[m][1] for m in present], axis=0)
e0E, a0E = pe.compute_eer(gE, iE)[:2]

best = min(present, key=lambda m: pe.compute_eer(*scores[m])[:2][0])
gB, iB = scores[best]

esE = np.empty(N_BOOT); asE = np.empty(N_BOOT); diff = np.empty(N_BOOT)
for b in range(N_BOOT):
    gidx = rng.integers(0, len(gE), len(gE))
    iidx = rng.integers(0, len(iE), len(iE))
    eE, aE = eer_auc_fast(gE[gidx], iE[iidx]); esE[b] = eE; asE[b] = aE
    eB, _ = eer_auc_fast(gB[gidx], iB[iidx])
    diff[b] = eB - eE                          # >0 means ensemble better (lower EER)

s_eE, s_aE = esE.std(ddof=1), asE.std(ddof=1)
print(f"\n== ENSEMBLE (mean-cosine fusion of {len(present)} models) ==")
print(f"EER {e0E:.2f} [{esE.mean():.2f}+/-{s_eE:.2f}]  95% CI "
      f"[{np.percentile(esE,2.5):.2f}, {np.percentile(esE,97.5):.2f}]")
print(f"AUC {a0E:.4f} [{asE.mean():.4f}+/-{s_aE:.4f}]")

p_boot = 2 * min((diff <= 0).mean(), (diff >= 0).mean())
print(f"\nEnsemble vs best single ({best}): mean EER gain {diff.mean():.2f} pts, "
      f"95% CI [{np.percentile(diff,2.5):.2f}, {np.percentile(diff,97.5):.2f}], "
      f"bootstrap two-sided p = {p_boot:.4f}  (N_BOOT={N_BOOT})")

print("\n--- LaTeX (per-model rows) ---")
for r in latex_rows: print(r)
print(f"\\textbf{{Ensemble}} & $\\mathbf{{{e0E:.2f} \\pm {s_eE:.2f}}}$ & "
      f"$\\mathbf{{{a0E:.4f} \\pm {s_aE:.4f}}}$ \\\\")

print("\n--- suggested methods sentence ---")
print(f"Uncertainty is reported as the mean and standard deviation of a "
      f"non-parametric bootstrap (N={N_BOOT} resamples) over the verification "
      f"pairs; the ensemble improves EER over the best single model by "
      f"{diff.mean():.2f} points (95% CI "
      f"[{np.percentile(diff,2.5):.2f}, {np.percentile(diff,97.5):.2f}], "
      f"bootstrap p={p_boot:.4f}).")